# Retail Data Analytics with PySpark

In [0]:
df = (
    spark.read
    .format("jdbc")
    .option("url", "jdbc:postgresql://34.130.111.160:5432/postgres")
    .option("dbtable", "public.retail")
    .option("user", "postgres")
    .option("password", "password")
    .option("driver", "org.postgresql.Driver")
    .load()
)
display(df.limit(10))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


In [0]:
df.show(10);

+----------+----------+--------------------+--------+-------------------+----------+-----------+--------------+
|invoice_no|stock_code|         description|quantity|       invoice_date|unit_price|customer_id|       country|
+----------+----------+--------------------+--------+-------------------+----------+-----------+--------------+
|    489434|     85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00|      6.95|    13085.0|United Kingdom|
|    489434|    79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00|      6.75|    13085.0|United Kingdom|
|    489434|    79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00|      6.75|    13085.0|United Kingdom|
|    489434|     22041|RECORD FRAME 7" S...|      48|2009-12-01 07:45:00|       2.1|    13085.0|United Kingdom|
|    489434|     21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00|      1.25|    13085.0|United Kingdom|
|    489434|     22064|PINK DOUGHNUT TRI...|      24|2009-12-01 07:45:00|      1.65|    13085.0|United K

In [0]:
df.printSchema();


root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- unit_price: float (nullable = true)
 |-- customer_id: float (nullable = true)
 |-- country: string (nullable = true)



# Clean and standardize the dataframe

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df = (
    df.withColumn("invoice_no", F.col("invoice_no").cast("string"))
      .withColumn("stock_code", F.col("stock_code").cast("string"))
      .withColumn("description", F.col("description").cast("string"))
      .withColumn("quantity", F.col("quantity").cast("int"))
      .withColumn("invoice_date", F.col("invoice_date").cast("timestamp"))
      .withColumn("unit_price", F.col("unit_price").cast("double"))
      .withColumn("customer_id", F.col("customer_id").cast("int"))
      .withColumn("country", F.col("country").cast("string"))
)

In [0]:
display(df.limit(10))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.949999809265137,13085,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.0999999046325684,13085,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.649999976158142,13085,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.949999809265137,13085,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.549999952316284,13085,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085,United Kingdom


In [0]:
df.printSchema()

root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)



## Simple null check in PySpark

In [0]:
from pyspark.sql import functions as F

null_check_df = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_check_df)

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,0,4382,0,0,0,243007,0


In [0]:
# Remove rows where description is null
df = df.dropna(subset=["description"])

display(df.limit(10))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,sales_amount,year_month,month_year
489556,21252,SET OF MEADOW FLOWER STICKERS,2,2009-12-01T12:47:00.000Z,2.950000047683716,15719,United Kingdom,5.900000095367432,200912,Dec 2009
489556,84946,ANTIQUE SILVER TEA GLASS ETCHED,6,2009-12-01T12:47:00.000Z,1.25,15719,United Kingdom,7.5,200912,Dec 2009
489556,85231B,CINAMMON SET OF 9 T-LIGHTS,2,2009-12-01T12:47:00.000Z,0.8500000238418579,15719,United Kingdom,1.7000000476837158,200912,Dec 2009
489556,85231L,LAVENDER SCENTED SET/9 T-LIGHTS,1,2009-12-01T12:47:00.000Z,0.8500000238418579,15719,United Kingdom,0.8500000238418579,200912,Dec 2009
489556,20750,RED/WHITE DOT MINI CASES,1,2009-12-01T12:47:00.000Z,7.949999809265137,15719,United Kingdom,7.949999809265137,200912,Dec 2009
489556,82482,WOODEN PICTURE FRAME WHITE FINISH,4,2009-12-01T12:47:00.000Z,2.549999952316284,15719,United Kingdom,10.199999809265137,200912,Dec 2009
489556,85152,HAND OVER THE CHOCOLATE SIGN,4,2009-12-01T12:47:00.000Z,2.0999999046325684,15719,United Kingdom,8.399999618530273,200912,Dec 2009
489556,21174,POTTERING IN THE SHED METAL SIGN,2,2009-12-01T12:47:00.000Z,1.9500000476837158,15719,United Kingdom,3.9000000953674316,200912,Dec 2009
489556,20665,RED SPOTTY PURSE,1,2009-12-01T12:47:00.000Z,2.950000047683716,15719,United Kingdom,2.950000047683716,200912,Dec 2009
489556,82486,WOOD S/3 CABINET ANT WHITE FINISH,2,2009-12-01T12:47:00.000Z,7.949999809265137,15719,United Kingdom,15.899999618530273,200912,Dec 2009


## Added a sales amount column

In [0]:
df = df.withColumn("sales_amount", F.col("quantity") * F.col("unit_price"))
display(df.limit(10))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,sales_amount
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.949999809265137,13085,United Kingdom,83.39999771118164
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,United Kingdom,81.0
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,United Kingdom,81.0
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.0999999046325684,13085,United Kingdom,100.79999542236328
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085,United Kingdom,30.0
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.649999976158142,13085,United Kingdom,39.59999942779541
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085,United Kingdom,30.0
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.949999809265137,13085,United Kingdom,59.49999809265137
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.549999952316284,13085,United Kingdom,30.59999942779541
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085,United Kingdom,45.0


## Added year_month column and month_year column

In [0]:
df = (
    df.withColumn("year_month", F.date_format("invoice_date", "yyyyMM").cast("int"))
      .withColumn("month_year", F.date_format("invoice_date", "MMM yyyy"))
)
display(df.limit(10))

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,sales_amount,year_month,month_year
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.949999809265137,13085,United Kingdom,83.39999771118164,200912,Dec 2009
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,United Kingdom,81.0,200912,Dec 2009
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,United Kingdom,81.0,200912,Dec 2009
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.0999999046325684,13085,United Kingdom,100.79999542236328,200912,Dec 2009
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085,United Kingdom,30.0,200912,Dec 2009
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.649999976158142,13085,United Kingdom,39.59999942779541,200912,Dec 2009
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085,United Kingdom,30.0,200912,Dec 2009
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.949999809265137,13085,United Kingdom,59.49999809265137,200912,Dec 2009
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.549999952316284,13085,United Kingdom,30.59999942779541,200912,Dec 2009
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085,United Kingdom,45.0,200912,Dec 2009


# Total Invoice Amount Distribution

In [0]:
df_positive = df.filter(
    (F.col("quantity") > 0) & (F.col("unit_price") > 0)
)

In [0]:
invoice_amount_df = (
    df_positive
    .groupBy("invoice_no")
    .agg(F.round(F.sum("sales_amount"), 2).alias("invoice_amount"))
    .orderBy("invoice_amount")
)


In [0]:
invoice_amount_df.select(
    F.min("invoice_amount").alias("min_invoice_amount"),
    F.max("invoice_amount").alias("max_invoice_amount"),
    F.mean("invoice_amount").alias("mean_invoice_amount"),
    F.expr("percentile(invoice_amount, 0.5)").alias("median_invoice_amount")
).show()

+------------------+------------------+-------------------+---------------------+
|min_invoice_amount|max_invoice_amount|mean_invoice_amount|median_invoice_amount|
+------------------+------------------+-------------------+---------------------+
|              0.19|         168469.59|  523.3037604171865|              304.315|
+------------------+------------------+-------------------+---------------------+



In [0]:
invoice_amount_mode_df = (
    invoice_amount_df.groupBy("invoice_amount")
    .count()
    .orderBy(F.desc("invoice_amount"))
)

display(invoice_amount_mode_df.limit(10))

invoice_amount,count
168469.59,1
77183.6,1
52940.94,1
50653.91,1
49844.99,1
45332.97,1
44051.6,1
38970.0,1
33167.8,1
31770.98,1


In [0]:
q85 = invoice_amount_df.approxQuantile("invoice_amount", [0.85], 0.01)[0]
print(q85)

692.96


In [0]:
invoice_amount_85_df = invoice_amount_df.filter(F.col("invoice_amount") <= q85)
display(invoice_amount_85_df.limit(10))

invoice_no,invoice_amount
528127,0.19
570554,0.38
567869,0.4
529767,0.42
507293,0.42
502731,0.42
539441,0.42
518991,0.42
532608,0.5
519123,0.55


# Monthly Placed and Canceled Orders

In [0]:
# Monthly canceled orders
monthly_canceled_orders_df = (
    df.filter(F.col("invoice_no").startswith("C"))
      .groupBy("year_month", "month_year")
      .agg(F.countDistinct("invoice_no").alias("canceled_orders"))
)

# Monthly total unique invoices
monthly_total_orders_df = (
    df.groupBy("year_month", "month_year")
      .agg(F.countDistinct("invoice_no").alias("total_orders"))
)

# Monthly placed orders vs canceled orders
monthly_orders_df = (
    monthly_total_orders_df.join(
        monthly_canceled_orders_df,
        on=["year_month", "month_year"],
        how="left"
    )
    .fillna(0, subset=["canceled_orders"])
    .withColumn("canceled_orders", F.col("canceled_orders").cast("int"))
    .withColumn("placed_orders", F.col("total_orders") - 2 * F.col("canceled_orders"))
    .select("year_month", "month_year", "placed_orders", "canceled_orders")
    .orderBy("year_month")
)

display(monthly_orders_df.limit(10))

year_month,month_year,placed_orders,canceled_orders
200912,Dec 2009,1300,401
201001,Jan 2010,811,300
201002,Feb 2010,979,240
201003,Mar 2010,1301,407
201004,Apr 2010,1174,304
201005,May 2010,1128,407
201006,Jun 2010,1327,357
201007,Jul 2010,1244,344
201008,Aug 2010,1172,273
201009,Sep 2010,1498,371


# Monthly Sales

In [0]:
monthly_sales_df = (
    df.filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))
      .groupBy("year_month", "month_year")
      .agg(F.round(F.sum("sales_amount"), 2).alias("monthly_sales"))
      .orderBy("year_month")
)

display(monthly_sales_df.limit(10))

year_month,month_year,monthly_sales
200912,Dec 2009,825685.76
201001,Jan 2010,652708.5
201002,Feb 2010,553713.3
201003,Mar 2010,833570.13
201004,Apr 2010,681528.99
201005,May 2010,659858.86
201006,Jun 2010,752270.14
201007,Jul 2010,650712.94
201008,Aug 2010,697274.91
201009,Sep 2010,924333.01


Databricks visualization. Run in Databricks to view.

# Monthly Sales Growth

In [0]:
sales_window = Window.orderBy("year_month")

monthly_sales_growth_df = (
    monthly_sales_df
      .withColumn("previous_month_sales", F.lag("monthly_sales").over(sales_window))
      .withColumn(
          "growth_rate",
          F.round(
              ((F.col("monthly_sales") - F.col("previous_month_sales")) / F.col("previous_month_sales")) * 100,
              2
          )
      )
      .select("year_month", "month_year", "monthly_sales", "previous_month_sales", "growth_rate")
      .orderBy("year_month")
)

display(monthly_sales_growth_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


year_month,month_year,monthly_sales,previous_month_sales,growth_rate
200912,Dec 2009,825685.76,null,null
201001,Jan 2010,652708.5,825685.76,-20.95
201002,Feb 2010,553713.3,652708.5,-15.17
201003,Mar 2010,833570.13,553713.3,50.54
201004,Apr 2010,681528.99,833570.13,-18.24
201005,May 2010,659858.86,681528.99,-3.18
201006,Jun 2010,752270.14,659858.86,14.0
201007,Jul 2010,650712.94,752270.14,-13.5
201008,Aug 2010,697274.91,650712.94,7.16
201009,Sep 2010,924333.01,697274.91,32.56


# Monthly Active Users

In [0]:
monthly_active_users_df = (
    df.filter(F.col("customer_id").isNotNull())
      .groupBy("year_month", "month_year")
      .agg(F.countDistinct("customer_id").alias("active_users"))
      .orderBy("year_month")
)

display(monthly_active_users_df.limit(10))

year_month,month_year,active_users
200912,Dec 2009,1045
201001,Jan 2010,786
201002,Feb 2010,807
201003,Mar 2010,1111
201004,Apr 2010,998
201005,May 2010,1062
201006,Jun 2010,1095
201007,Jul 2010,988
201008,Aug 2010,964
201009,Sep 2010,1202


# New and Existing Users

In [0]:
customer_first_purchase_df = (
    df.filter(F.col("customer_id").isNotNull())
      .groupBy("customer_id")
      .agg(F.min("year_month").alias("first_purchase_month"))
      .withColumn(
          "first_purchase_month_year",
          F.date_format(F.to_date(F.col("first_purchase_month").cast("string"), "yyyyMM"), "MMM yyyy")
      )
)

In [0]:
customer_activity_df = (
    df.filter(F.col("customer_id").isNotNull())
      .select("customer_id", "year_month", "month_year")
      .distinct()
      .join(customer_first_purchase_df, on="customer_id", how="left")
      .withColumn(
          "user_type",
          F.when(F.col("year_month") == F.col("first_purchase_month"), "new")
           .otherwise("existing")
      )
      .orderBy("year_month")
)

In [0]:
monthly_new_vs_existing_users_df = (
    customer_activity_df.groupBy("year_month", "month_year")
      .pivot("user_type", ["new", "existing"])
      .agg(F.countDistinct("customer_id"))
      .fillna(0)
      .withColumn("new", F.col("new").cast("int"))
      .withColumn("existing", F.col("existing").cast("int"))
      .orderBy("year_month")
)

display(monthly_new_vs_existing_users_df.limit(10))

year_month,month_year,new,existing
200912,Dec 2009,1045,0
201001,Jan 2010,394,392
201002,Feb 2010,363,444
201003,Mar 2010,436,675
201004,Apr 2010,291,707
201005,May 2010,254,808
201006,Jun 2010,269,826
201007,Jul 2010,183,805
201008,Aug 2010,158,806
201009,Sep 2010,242,960


Databricks visualization. Run in Databricks to view.

# Finding RFM

In [0]:
rfm_base_df = (
    df.filter(F.col("customer_id").isNotNull())
)

In [0]:
max_date_row = rfm_base_df.select(F.max("invoice_date").alias("max_date")).collect()[0]
max_date = max_date_row["max_date"]

print("Max invoice date:", max_date)

Max invoice date: 2011-12-09 12:50:00


In [0]:
snapshot_date = F.date_add(F.lit(max_date), 1)

In [0]:
rfm_df = (
    rfm_base_df.groupBy("customer_id")
      .agg(
          F.max("invoice_date").alias("last_purchase_date"),
          F.countDistinct("invoice_no").alias("frequency"),
          F.round(F.sum("sales_amount"), 2).alias("monetary")
      )
      .withColumn("snapshot_date", snapshot_date)
      .withColumn("recency", F.datediff(F.col("snapshot_date"), F.col("last_purchase_date")))
      .select(
          "customer_id",
          "last_purchase_date",
          "snapshot_date",
          "recency",
          "frequency",
          "monetary"
      )
      .orderBy("customer_id")
)

display(rfm_df.limit(10))

customer_id,last_purchase_date,snapshot_date,recency,frequency,monetary
12346,2011-01-18T10:17:00.000Z,2011-12-10,326,17,-64.68
12347,2011-12-07T15:52:00.000Z,2011-12-10,3,8,5633.32
12348,2011-09-25T13:13:00.000Z,2011-12-10,76,5,2019.4
12349,2011-11-21T09:51:00.000Z,2011-12-10,19,5,4404.54
12350,2011-02-02T16:01:00.000Z,2011-12-10,311,1,334.4
12351,2010-11-29T15:23:00.000Z,2011-12-10,376,1,300.93
12352,2011-11-03T14:37:00.000Z,2011-12-10,37,13,1889.21
12353,2011-05-19T17:47:00.000Z,2011-12-10,205,2,406.76
12354,2011-04-21T13:11:00.000Z,2011-12-10,233,1,1079.4
12355,2011-05-09T13:49:00.000Z,2011-12-10,215,2,947.61


# RFM Segmentation

In [0]:
recency_quantiles = rfm_df.approxQuantile("recency", [0.2, 0.4, 0.6, 0.8], 0.01)
frequency_quantiles = rfm_df.approxQuantile("frequency", [0.2, 0.4, 0.6, 0.8], 0.01)
monetary_quantiles = rfm_df.approxQuantile("monetary", [0.2, 0.4, 0.6, 0.8], 0.01)

print("Recency quantiles:", recency_quantiles)
print("Frequency quantiles:", frequency_quantiles)
print("Monetary quantiles:", monetary_quantiles)

Recency quantiles: [18.0, 59.0, 186.0, 409.0]
Frequency quantiles: [1.0, 3.0, 5.0, 10.0]
Monetary quantiles: [251.52, 587.05, 1171.46, 2744.14]


In [0]:
rfm_scored_df = (
    rfm_df.withColumn(
        "r_score",
        F.when(F.col("recency") <= recency_quantiles[0], 5)
         .when(F.col("recency") <= recency_quantiles[1], 4)
         .when(F.col("recency") <= recency_quantiles[2], 3)
         .when(F.col("recency") <= recency_quantiles[3], 2)
         .otherwise(1)
    )
    .withColumn(
        "f_score",
        F.when(F.col("frequency") <= frequency_quantiles[0], 1)
         .when(F.col("frequency") <= frequency_quantiles[1], 2)
         .when(F.col("frequency") <= frequency_quantiles[2], 3)
         .when(F.col("frequency") <= frequency_quantiles[3], 4)
         .otherwise(5)
    )
    .withColumn(
        "m_score",
        F.when(F.col("monetary") <= monetary_quantiles[0], 1)
         .when(F.col("monetary") <= monetary_quantiles[1], 2)
         .when(F.col("monetary") <= monetary_quantiles[2], 3)
         .when(F.col("monetary") <= monetary_quantiles[3], 4)
         .otherwise(5)
    )
    .withColumn(
        "rfm_score",
        F.concat(
            F.col("r_score").cast("string"),
            F.col("f_score").cast("string"),
            F.col("m_score").cast("string")
        )
    )
)

display(rfm_scored_df.limit(10))

customer_id,last_purchase_date,snapshot_date,recency,frequency,monetary,r_score,f_score,m_score,rfm_score
12346,2011-01-18T10:17:00.000Z,2011-12-10,326,17,-64.68,2,5,1,251
12347,2011-12-07T15:52:00.000Z,2011-12-10,3,8,5633.32,5,4,5,545
12348,2011-09-25T13:13:00.000Z,2011-12-10,76,5,2019.4,3,3,4,334
12349,2011-11-21T09:51:00.000Z,2011-12-10,19,5,4404.54,4,3,5,435
12350,2011-02-02T16:01:00.000Z,2011-12-10,311,1,334.4,2,1,2,212
12351,2010-11-29T15:23:00.000Z,2011-12-10,376,1,300.93,2,1,2,212
12352,2011-11-03T14:37:00.000Z,2011-12-10,37,13,1889.21,4,5,4,454
12353,2011-05-19T17:47:00.000Z,2011-12-10,205,2,406.76,2,2,2,222
12354,2011-04-21T13:11:00.000Z,2011-12-10,233,1,1079.4,2,1,3,213
12355,2011-05-09T13:49:00.000Z,2011-12-10,215,2,947.61,2,2,3,223


In [0]:
rfm_segmented_df = (
    rfm_scored_df.withColumn(
        "segment",
        F.when(
            (F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4),
            "Champions"
        )
        .when(
            (F.col("r_score") >= 3) & (F.col("f_score") >= 3) & (F.col("m_score") >= 3),
            "Loyal Customers"
        )
        .when(
            (F.col("r_score") >= 4) & (F.col("f_score") <= 2),
            "New Customers"
        )
        .when(
            (F.col("r_score") <= 2) & (F.col("f_score") >= 3),
            "At Risk"
        )
        .when(
            (F.col("r_score") <= 2) & (F.col("f_score") <= 2) & (F.col("m_score") <= 2),
            "Lost"
        )
        .otherwise("Others")
    )
)

display(rfm_segmented_df.limit(10))

customer_id,last_purchase_date,snapshot_date,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
12346,2011-01-18T10:17:00.000Z,2011-12-10,326,17,-64.68,2,5,1,251,At Risk
12347,2011-12-07T15:52:00.000Z,2011-12-10,3,8,5633.32,5,4,5,545,Champions
12348,2011-09-25T13:13:00.000Z,2011-12-10,76,5,2019.4,3,3,4,334,Loyal Customers
12349,2011-11-21T09:51:00.000Z,2011-12-10,19,5,4404.54,4,3,5,435,Loyal Customers
12350,2011-02-02T16:01:00.000Z,2011-12-10,311,1,334.4,2,1,2,212,Lost
12351,2010-11-29T15:23:00.000Z,2011-12-10,376,1,300.93,2,1,2,212,Lost
12352,2011-11-03T14:37:00.000Z,2011-12-10,37,13,1889.21,4,5,4,454,Champions
12353,2011-05-19T17:47:00.000Z,2011-12-10,205,2,406.76,2,2,2,222,Lost
12354,2011-04-21T13:11:00.000Z,2011-12-10,233,1,1079.4,2,1,3,213,Others
12355,2011-05-09T13:49:00.000Z,2011-12-10,215,2,947.61,2,2,3,223,Others


In [0]:
rfm_segment_summary_df = (
    rfm_segmented_df.groupBy("segment")
      .agg(
          F.count("*").alias("customer_count"),
          F.round(F.avg("monetary"), 2).alias("avg_monetary_value")
      )
      .orderBy(F.desc("customer_count"))
)

display(rfm_segment_summary_df)

segment,customer_count,avg_monetary_value
Lost,1440,192.21
Champions,1261,9225.52
Loyal Customers,1006,2331.31
Others,960,773.79
New Customers,675,633.64
At Risk,600,2037.14
